# Feature extraction (shape, color, texture)

Reads `preprocessed_binary/manifest.csv` (keeping only rows with status `ok` or `skipped`) and, for each image,
loads its preprocessed image (`_preprocessed.png`: color-corrected and hair-free, no blur) and its mask (`_mask.png`)
to extract **412 features**. Saves **one CSV per split**, `features_train.csv` and `features_val.csv`, in
`features_binary_final/` (`OUT_DIR`; a new folder so the previous CSVs with 420 features are not overwritten).

The extraction functions live in `features/features.py` (imported here as `F`). The feature parameters are changed
there: `EXTENDED_SHAPE` (True), `KMEANS_K` (6), `RIDGE_THR` (0.02) and the Gabor bank (`GABOR_*`). With
`EXTENDED_SHAPE = False` the `shape_` family has 25 columns instead of 52, so the total is no longer 412.
This notebook only contains the paths, the parallel execution and a summary. Set `LIMIT` to a small number to test with a few images.

**Columns of the CSVs**
- First `name` (image name), then `class`, `subtype`, `label` (nevus=0, others=1), the quality flags `qc_seg_ok` and
  `qc_crop_fallback` (taken from the manifest's `seg_ok` and `crop_fallback`; NaN if the manifest does not have them),
  then the features, and finally `qc_errors` (errors raised during extraction, separated by `;`; empty if there were none).
  The `qc_*` columns are not features: do not use them for training.
- Features are prefixed `shape_`, `color_`, `tex_`, `gcolor_` or `gtex_` so you can run ablations by family.

**Feature families (412)**

| Prefix | Cols | Computed on | Contents |
|---|---|---|---|
| `shape_` | 52 | lesion mask | **Basic (25):** size and proportions (area fraction, perimeter, major/minor axis, extent, circularity, solidity, eccentricity, aspect ratio, rectangularity), asymmetry (overlap with the mirrored mask about both axes), radial signature (CV, range), concavities (number, max, mean) and edge sharpness (gradient and contrast in a band around the contour, overall and per 8 sectors). **Extended (27, `EXTENDED_SHAPE`):** 7 Hu moments, 10 Fourier descriptors, 6 curvature, 2 Feret diameters, fractal dimension and lacunarity |
| `color_` | 82 | lesion mask | 8 channels (R, G, B, S, V, L, a, b) x [7 statistics (mean, std, skew, kurtosis, p5, p50, p95) + lesion minus surrounding skin `color_rel_*`] = 64; circular hue statistics (3); K-means in Lab: number of colors, entropy, cluster spread (3); clinical color fractions (white, black, blue-gray, red, dark brown) + number of clinical colors (6); core minus periphery in L, a, b, S (4); color asymmetry between halves (2) |
| `tex_` | 104 | lesion mask | GLCM: 6 properties x 3 distances, averaged over 4 angles (18); uniform LBP (10); LNDP (11); Sato ridge filter for pigment network (3); LoG blobs for dots/globules (3); gray entropy and local std (2); **Gabor `tex_gabor_*` (57)**: 4 wavelengths x 6 orientations (mean and std = 48) + energy and anisotropy per wavelength (8) + fine/coarse ratio (1) |
| `gcolor_` | 70 | whole image | Same as `color_` without the lesion-vs-skin (`rel`) and core-vs-periphery columns (12 fewer) |
| `gtex_` | 104 | whole image | Same as `tex_` (including `gtex_lndp_*` and `gtex_gabor_*`) computed over the whole image |

Note: `color_rel_*` and the edge features of `shape_` also look slightly outside the lesion (a skin ring around it / a band along the contour).

**Failures and NaN**
- If a family fails, its columns stay as NaN and the error is recorded in `qc_errors`; the batch is not interrupted.
- If the mask is empty or degenerate (fewer than 50 px), `mask_empty` is recorded and only `gcolor_`, `gtex_` and `gtex_gabor_*` are computed; `shape_`, `color_`, `tex_` and `tex_gabor_*` stay as NaN.

In [ ]:
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from features import features as F

# ---------------- Configuration ----------------
ROOT = Path(r"C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG")
PROC_DIR = ROOT / "preprocessed_binary"
OUT_DIR = ROOT / "features_binary_final"    # new folder so the previous CSVs (420 features) are not overwritten
N_JOBS = 4                           # cores (-1 = all, -2 = all but one)
LIMIT = None                         # e.g. 20 to test with a few images; None = all

In [ ]:
manifest = pd.read_csv(PROC_DIR / "manifest.csv")
manifest = manifest[manifest["status"].isin(["ok", "skipped"])].reset_index(drop=True)

if LIMIT:
    manifest = manifest.groupby("split").head(LIMIT).reset_index(drop=True)
print("processed:", PROC_DIR)
print(manifest.groupby(["split", "class"]).size().to_string())

## Extraction (parallel)

In [ ]:
gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(F.extract_one)(r) for r in manifest.to_dict("records"))
df = pd.DataFrame([r for r in tqdm(gen, total=len(manifest))])

meta = ["name", "class", "subtype", "label", "qc_seg_ok", "qc_crop_fallback"]
feat_cols = [c for p in F.FEATURE_PREFIXES for c in df.columns if c.startswith(p)]
df = df[meta + feat_cols + ["qc_errors"]]
df["_split"] = df["name"].map(manifest.set_index("name")["split"])
print(f"\n{df.shape[0]} images, {len(feat_cols)} features")

## Saving: one CSV per split

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ["train", "val"]:
    part = df[df["_split"] == split].drop(columns="_split")
    if len(part) == 0:
        continue
    path = OUT_DIR / f"features_{split}.csv"
    part.to_csv(path, index=False)
    print(f"{path}  ->  {part.shape[0]} rows x {part.shape[1]} columns")

# per-family summary (expected: shape 52, color 82, tex 104, gcolor 70, gtex 104)
nan_pct = df[feat_cols].isna().mean()
for p in ["shape", "color", "tex", "gcolor", "gtex"]:
    cols_p = [c for c in feat_cols if c.startswith(p + "_")]
    if cols_p:
        print(f"{p:6s} {len(cols_p):3d} columns | mean NaN: {100 * nan_pct[cols_p].mean():.2f}%")
    else:
        print(f"{p:6s}   0 columns  <-- the family failed on every image! check qc_errors")
bad = df[df["qc_errors"] != ""]
print(f"Images with at least one error: {len(bad)}")
if len(bad):
    print(bad["qc_errors"].str.split(";").explode().str.replace(r"^(\w+:\w+@\w+:L\d+).*", r"\1", regex=True)
          .value_counts().head(10).to_string())          # most frequent errors (type, function and line)
    print(bad[["name", "qc_errors"]].head(5).to_string(index=False))